<a href="https://colab.research.google.com/github/leekx006/Lyapis2.OHG-Latin-Alignment/blob/main/Sphiberta_saved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COMPLETE MORNING WORKFLOW
# ============================================================

# 1. Mount Drive and Load Data
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')

print("=" * 60)
print("LOADING DATA")
print("=" * 60)

datasets = {
    'Tatian': '1IgL9l0b1iVLDZHApNrVqYBjXj5mr5S58',
    'Benedictine Rule': '1nFjBTWpRMaUW5xkC4usm_FrR3_jOIjkK',
    'Isidore of Seville': '1TU1HdmKrRUpwrN8tDnLzTGlyJ9uj03UH',
    'Physiologus': '1xtuNhLFnlQAMfyx_5jzT0T8IOxEaWJuC'
}

def clean_corpus(df, name):
    original_count = len(df)
    df_clean = df[
        (df['Target clause'].notna()) &
        (df['Target clause'] != '\\N') &
        (df['Target clause'].astype(str).str.strip() != '') &
        (df['Source clause'].notna()) &
        (df['Source clause'].astype(str).str.strip() != '')
    ].copy()
    df_clean['Source clause'] = df_clean['Source clause'].astype(str).str.strip()
    df_clean['Target clause'] = df_clean['Target clause'].astype(str).str.strip()
    return df_clean

all_dataframes = {}
for name, file_id in datasets.items():
    export_url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"
    df = pd.read_excel(export_url)
    all_dataframes[name] = clean_corpus(df, name)

all_pairs = []
for name, df in all_dataframes.items():
    df['source_text'] = name
    all_pairs.append(df[['Source clause', 'Target clause', 'source_text']])

combined_df = pd.concat(all_pairs, ignore_index=True)
print(f"✅ Loaded {len(combined_df)} parallel pairs")

# 2. Load Models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

print("\n" + "=" * 60)
print("LOADING PRE-TRAINED MODELS")
print("=" * 60)

sphilberta = SentenceTransformer('bowphs/SPhilBERTa')
mbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
mbert_model = AutoModel.from_pretrained('bert-base-multilingual-cased')
print("✅ Models loaded")

# 3. Split and Encode Data
train_df, temp_df = train_test_split(combined_df, test_size=0.2, random_state=42,
                                      stratify=combined_df['source_text'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"\n📊 Split: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test")

print("\n🔤 Encoding with SPhilBERTa...")
latin_train = sphilberta.encode(train_df['Source clause'].tolist(),
                                convert_to_tensor=True, show_progress_bar=True)
latin_val = sphilberta.encode(val_df['Source clause'].tolist(),
                              convert_to_tensor=True, show_progress_bar=False)
latin_test = sphilberta.encode(test_df['Source clause'].tolist(),
                               convert_to_tensor=True, show_progress_bar=False)

# Clone for training
latin_train = latin_train.clone().detach()
latin_val = latin_val.clone().detach()
latin_test = latin_test.clone().detach()

print("🔤 Encoding with mBERT...")
def encode_ohg_batch(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = mbert_tokenizer(batch, return_tensors="pt", truncation=True,
                                 max_length=128, padding=True)
        with torch.no_grad():
            outputs = mbert_model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :])
    return torch.cat(embeddings, dim=0)

ohg_train = encode_ohg_batch(train_df['Target clause'].tolist())
ohg_val = encode_ohg_batch(val_df['Target clause'].tolist())
ohg_test = encode_ohg_batch(test_df['Target clause'].tolist())

print("✅ All data encoded")

# 4. Define Architecture
class ImprovedProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=768, hidden_dim=1024, dropout=0.2):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.projection(x)

class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, ohg_embeddings, latin_embeddings):
        ohg_norm = F.normalize(ohg_embeddings, p=2, dim=1)
        latin_norm = F.normalize(latin_embeddings, p=2, dim=1)
        logits = torch.matmul(ohg_norm, latin_norm.T) / self.temperature
        batch_size = ohg_embeddings.size(0)
        labels = torch.arange(batch_size).to(ohg_embeddings.device)
        loss = F.cross_entropy(logits, labels)
        return loss

# 5. Training Function
def train_model(loss_type='contrastive', verbose=True):
    """Train alignment model with specified loss"""

    projection = ImprovedProjection(hidden_dim=1024, dropout=0.1)
    optimizer = optim.AdamW(projection.parameters(), lr=0.0001, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    if loss_type == 'contrastive':
        loss_fn = ContrastiveLoss(temperature=0.07)
    else:
        loss_fn = nn.MSELoss()

    if verbose:
        print(f"\n{'='*60}")
        print(f"Training with {loss_type.upper()} loss")
        print(f"{'='*60}")

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(100):
        projection.train()
        train_loss = 0
        num_batches = 0

        for i in range(0, len(train_df), 64):
            batch_ohg = ohg_train[i:i+64]
            batch_latin = latin_train[i:i+64]

            if batch_ohg.size(0) < 2:
                continue

            projected_ohg = projection(batch_ohg)
            loss = loss_fn(projected_ohg, batch_latin)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(projection.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            num_batches += 1

        avg_train_loss = train_loss / num_batches

        projection.eval()
        with torch.no_grad():
            val_projected = projection(ohg_val)
            val_loss = loss_fn(val_projected, latin_val).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = projection.state_dict().copy()
        else:
            patience_counter += 1

        if verbose and ((epoch + 1) % 10 == 0 or patience_counter == 0):
            print(f"Epoch {epoch+1:3d} | Train: {avg_train_loss:.4f} | Val: {val_loss:.4f}")

        if patience_counter >= 15:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}")
            break

    projection.load_state_dict(best_model_state)

    # Evaluate
    projection.eval()
    with torch.no_grad():
        test_projected = projection(ohg_test)
        similarities = []
        for i in range(len(test_df)):
            sim = cosine_similarity(
                test_projected[i:i+1].cpu().numpy(),
                latin_test[i:i+1].cpu().numpy()
            )[0][0]
            similarities.append(sim)

    mean_sim = np.mean(similarities)

    if verbose:
        print(f"✅ Training complete!")
        print(f"   Mean similarity: {mean_sim:.4f}")

    return projection, best_model_state, mean_sim, similarities

# 6. Train Both Models
print("\n" + "=" * 60)
print("TRAINING BOTH MODELS")
print("=" * 60)

print("\n[1/2] Training MSE model...")
projection_mse, state_mse, sim_mse, sims_mse = train_model('mse')

print("\n[2/2] Training Contrastive model...")
projection_contrastive, state_contrastive, sim_contrastive, sims_contrastive = train_model('contrastive')

# 7. Save Models
import os
save_dir = '/content/drive/MyDrive/OHG_Models'
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'model_state_dict': state_mse,
    'mean_similarity': sim_mse,
    'config': {'loss': 'MSE', 'teacher': 'SPhilBERTa'}
}, f'{save_dir}/ohg_alignment_mse.pt')

torch.save({
    'model_state_dict': state_contrastive,
    'mean_similarity': sim_contrastive,
    'config': {'loss': 'Contrastive', 'teacher': 'SPhilBERTa'}
}, f'{save_dir}/ohg_alignment_contrastive.pt')

print(f"\n{'='*60}")
print("✅ BOTH MODELS SAVED")
print(f"{'='*60}")
print(f"   MSE Model:         {sim_mse:.4f} → {save_dir}/ohg_alignment_mse.pt")
print(f"   Contrastive Model: {sim_contrastive:.4f} → {save_dir}/ohg_alignment_contrastive.pt")
print(f"   Improvement:       {sim_contrastive - sim_mse:+.4f}")
print(f"{'='*60}")

Mounted at /content/drive
LOADING DATA
✅ Loaded 4208 parallel pairs

LOADING PRE-TRAINED MODELS


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Models loaded

📊 Split: 3366 train, 421 val, 421 test

🔤 Encoding with SPhilBERTa...


Batches:   0%|          | 0/106 [00:00<?, ?it/s]

🔤 Encoding with mBERT...
✅ All data encoded

TRAINING BOTH MODELS

[1/2] Training MSE model...

Training with MSE loss
Epoch   1 | Train: 0.0480 | Val: 0.0165
Epoch   2 | Train: 0.0266 | Val: 0.0157
Epoch   3 | Train: 0.0233 | Val: 0.0155
Epoch   4 | Train: 0.0217 | Val: 0.0154
Epoch   5 | Train: 0.0206 | Val: 0.0153
Epoch   6 | Train: 0.0196 | Val: 0.0152
Epoch   7 | Train: 0.0186 | Val: 0.0151
Epoch   8 | Train: 0.0178 | Val: 0.0150
Epoch   9 | Train: 0.0171 | Val: 0.0149
Epoch  10 | Train: 0.0166 | Val: 0.0148
Epoch  11 | Train: 0.0163 | Val: 0.0147
Epoch  12 | Train: 0.0160 | Val: 0.0146
Epoch  13 | Train: 0.0158 | Val: 0.0145
Epoch  14 | Train: 0.0156 | Val: 0.0145
Epoch  15 | Train: 0.0154 | Val: 0.0144
Epoch  16 | Train: 0.0152 | Val: 0.0143
Epoch  17 | Train: 0.0150 | Val: 0.0143
Epoch  18 | Train: 0.0149 | Val: 0.0142
Epoch  19 | Train: 0.0147 | Val: 0.0142
Epoch  20 | Train: 0.0146 | Val: 0.0141
Epoch  21 | Train: 0.0144 | Val: 0.0141
Epoch  22 | Train: 0.0143 | Val: 0.0141
E

In [ ]:
print("=" * 60)
print("TESTING OTFRID PARAPHRASES")
print("=" * 60)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Otfrid data
print("\n📥 Loading Otfrid paraphrase data...")
otfrid_file_id = '1PpmguwusrKwtEYuNeceAhIxznWfLSMQ0'
export_url = f"https://docs.google.com/spreadsheets/d/{otfrid_file_id}/export?format=xlsx"
otfrid_df = pd.read_excel(export_url)

print(f"✅ Loaded {len(otfrid_df)} Otfrid pairs")
print(f"Columns: {list(otfrid_df.columns)}")
print(f"\nSample:")
print(otfrid_df.head(3))

# 2. Load the trained MSE model
print("\n📂 Loading trained OHG alignment model...")

# Define architecture (same as training)
class ImprovedProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=768, hidden_dim=1024, dropout=0.2):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.projection(x)

# Create model and load weights
projection = ImprovedProjection(hidden_dim=1024, dropout=0.1)
checkpoint = torch.load('/content/drive/MyDrive/OHG_Models/ohg_alignment_mse.pt', weights_only=False)
projection.load_state_dict(checkpoint['model_state_dict'])
projection.eval()

print(f"✅ Model loaded (trained similarity: {checkpoint['mean_similarity']:.4f})")

# 3. Load pre-trained models for encoding
print("\n📥 Loading SPhilBERTa and mBERT...")
sphilberta = SentenceTransformer('bowphs/SPhilBERTa')
mbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
mbert_model = AutoModel.from_pretrained('bert-base-multilingual-cased')
print("✅ Models loaded")

# 4. Identify Latin and OHG columns
print("\n🔍 Identifying columns...")
# Let's see the actual column names first
for col in otfrid_df.columns:
    print(f"  Column: '{col}'")
    print(f"    Sample: {otfrid_df[col].iloc[0]}")
    print()

# You'll need to tell me which columns contain Latin and OHG
# For now, I'll assume similar structure to other datasets
# Adjust these based on what we see above:
latin_col = otfrid_df.columns[1]  # Probably second column?
ohg_col = otfrid_df.columns[3]    # Probably fourth column?

print(f"\n📊 Using columns:")
print(f"  Latin: '{latin_col}'")
print(f"  OHG: '{ohg_col}'")

# 5. Encode Otfrid texts
print("\n🔤 Encoding Otfrid texts...")

# Filter DataFrame to include only rows with valid Latin and OHG clauses
valid_pairs_df = otfrid_df.dropna(subset=[latin_col, ohg_col]).copy()

latin_texts = valid_pairs_df[latin_col].tolist()
ohg_texts = valid_pairs_df[ohg_col].tolist()

# Encode Latin with SPhilBERTa
latin_embeddings = sphilberta.encode(latin_texts, convert_to_tensor=True, show_progress_bar=False)
latin_embeddings = latin_embeddings.clone().detach()

# Encode OHG with mBERT
def encode_ohg_batch(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = mbert_tokenizer(batch, return_tensors="pt", truncation=True,
                                 max_length=128, padding=True)
        with torch.no_grad():
            outputs = mbert_model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :])
    return torch.cat(embeddings, dim=0)

ohg_embeddings = encode_ohg_batch(ohg_texts)

# Ensure lengths match after encoding
if len(latin_embeddings) != len(ohg_embeddings):
    raise ValueError("Mismatch in lengths of Latin and OHG embeddings after encoding. This should not happen with the filtered DataFrame.")

print(f"✅ Encoded {len(latin_embeddings)} pairs (after dropping NaNs)")

# 6. Test alignment
print("\n🧪 Testing alignment on Otfrid paraphrases...")

with torch.no_grad():
    # Project OHG embeddings to Latin space
    projected_ohg = projection(ohg_embeddings)

    # Calculate similarities
    similarities = []
    baseline_similarities = []

    for i in range(len(projected_ohg)):
        # With alignment
        sim_aligned = cosine_similarity(
            projected_ohg[i:i+1].cpu().numpy(),
            latin_embeddings[i:i+1].cpu().numpy()
        )[0][0]
        similarities.append(sim_aligned)

        # Baseline (no alignment)
        sim_baseline = cosine_similarity(
            ohg_embeddings[i:i+1].cpu().numpy(),
            latin_embeddings[i:i+1].cpu().numpy()
        )[0][0]
        baseline_similarities.append(sim_baseline)

# 7. Results
print(f"\n{'='*60}")
print("📊 OTFRID PARAPHRASE TEST RESULTS")
print(f"{'='*60}")

print(f"\n📈 Overall Performance:")
print(f"   Baseline (no alignment):  {np.mean(baseline_similarities):.4f} ± {np.std(baseline_similarities):.4f}")
print(f"   With alignment:           {np.mean(similarities):.4f} ± {np.std(similarities):.4f}")
print(f"   Improvement:              {np.mean(similarities) - np.mean(baseline_similarities):+.4f}")

print(f"\n📊 Comparison to Literal Translations:")
print(f"   Literal translations (Tatian, etc.): 0.7735")
print(f"   Paraphrases (Otfrid):                 {np.mean(similarities):.4f}")
print(f"   Difference:                           {np.mean(similarities) - 0.7735:+.4f}")

# Show some examples
print(f"\n📝 Sample Results:")
print(f"{'='*60}")

# Create results dataframe from the filtered dataframe
results_df = valid_pairs_df.copy()
results_df['similarity'] = similarities
results_df['baseline'] = baseline_similarities

# Sort by similarity to see best and worst
results_sorted = results_df.sort_values('similarity', ascending=False)

print("\n🏆 Best Matches (Top 5):")
for idx, row in results_sorted.head(5).iterrows():
    print(f"\nSimilarity: {row['similarity']:.4f}")
    print(f"  Latin: {row[latin_col][:80]}...")
    print(f"  OHG:   {row[ohg_col][:80]}...")

print("\n\n⚠️  Worst Matches (Bottom 5):")
for idx, row in results_sorted.tail(5).iterrows():
    print(f"\nSimilarity: {row['similarity']:.4f}")
    print(f"  Latin: {row[latin_col][:80]}...")
    print(f"  OHG:   {row[ohg_col][:80]}...")

# Statistics
print(f"\n{'='*60}")
print("📈 DETAILED STATISTICS")
print(f"{'='*60}")
print(f"Number of pairs:      {len(similarities)}")
print(f"Mean similarity:      {np.mean(similarities):.4f}")
print(f"Median similarity:    {np.median(similarities):.4f}")
print(f"Std deviation:        {np.std(similarities):.4f}")
print(f"Min similarity:       {np.min(similarities):.4f}")
print(f"Max similarity:       {np.max(similarities):.4f}")
print(f"\nPairs above 0.70:     {sum(s > 0.70 for s in similarities)} ({sum(s > 0.70 for s in similarities)/len(similarities)*100:.1f}%)")
print(f"Pairs above 0.60:     {sum(s > 0.60 for s in similarities)} ({sum(s > 0.60 for s in similarities)/len(similarities)*100:.1f}%)")
print(f"Pairs above 0.50:     {sum(s > 0.50 for s in similarities)} ({sum(s > 0.50 for s in similarities)/len(similarities)*100:.1f}%)")

# Interpretation
print(f"\n{'='*60}")
print("💡 INTERPRETATION")
print(f"{'='*60}")

mean_sim = np.mean(similarities)
if mean_sim > 0.70:
    print("✅ EXCELLENT: Model handles paraphrases nearly as well as literal translations!")
    print("   → Your literal-trained model generalizes to paraphrastic texts")
    print("   → No need to add Otfrid to training data")
elif mean_sim > 0.60:
    print("✅ GOOD: Model recognizes paraphrases with moderate success")
    print("   → Some generalization from literal to paraphrastic")
    print("   → Consider adding paraphrases to training for marginal improvement")
elif mean_sim > 0.50:
    print("⚠️  MODERATE: Model struggles somewhat with paraphrases")
    print("   → Trained on literal translations, less effective on loose adaptations")
    print("   → Adding Otfrid to training could help")
else:
    print("❌ POOR: Model fails on paraphrases")
    print("   → Too literal - doesn't recognize semantic equivalence")
    print("   → Definitely add paraphrastic training data")

print(f"\n{'='*60}")

TESTING OTFRID PARAPHRASES

📥 Loading Otfrid paraphrase data...
✅ Loaded 33 Otfrid pairs
Columns: ['Source clause number', 'Source clause', 'Target clause number', 'Target clause', 'Relation_Type']

Sample:
   Source clause number                              Source clause  \
0                     1                       Ne timeas, Zacharia,   
1                     2        quoniam exaudita est deprecatio tua   
2                     3  et uxor tua Elysabeth pariet tibi filium,   

   Target clause number                                      Target clause  \
0                     1                           Ni fórihti thir, bíscof,   
1                     2      wanta ist gibét thinaz fon drúhtine gihórtaz,   
2                     3  Joh áltquéna thinu ist thir kind berantu, sún ...   

  Relation_Type  
0    adaptation  
1    adaptation  
2    adaptation  

📂 Loading trained OHG alignment model...
✅ Model loaded (trained similarity: 0.7812)

📥 Loading SPhilBERTa and mBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bowphs/SPhilBERTa
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Models loaded

🔍 Identifying columns...
  Column: 'Source clause number'
    Sample: 1

  Column: 'Source clause'
    Sample: Ne timeas, Zacharia,

  Column: 'Target clause number'
    Sample: 1

  Column: 'Target clause'
    Sample: Ni fórihti thir, bíscof,

  Column: 'Relation_Type'
    Sample: adaptation


📊 Using columns:
  Latin: 'Source clause'
  OHG: 'Target clause'

🔤 Encoding Otfrid texts...
✅ Encoded 30 pairs (after dropping NaNs)

🧪 Testing alignment on Otfrid paraphrases...

📊 OTFRID PARAPHRASE TEST RESULTS

📈 Overall Performance:
   Baseline (no alignment):  0.0217 ± 0.0218
   With alignment:           0.7427 ± 0.0697
   Improvement:              +0.7210

📊 Comparison to Literal Translations:
   Literal translations (Tatian, etc.): 0.7735
   Paraphrases (Otfrid):                 0.7427
   Difference:                           -0.0308

📝 Sample Results:

🏆 Best Matches (Top 5):

Similarity: 0.8908
  Latin: et misus sum ad te...
  OHG:   Sánt er mih fon hímile,...

Similar

In [ ]:
# The projection layer and models are already loaded
# Just test your specific examples:

test_pairs = [
    ('quoniam exaudita est deprecatio tua',
     'wanta ist gibét thinaz fon drúhtine gihórtaz'),
    ('Et dixit Zacharias ad angelum',
     'Thó sprah ther bíscof, harto fóraht er mo thoh')
]

for latin, ohg in test_pairs:
    # Encode
    latin_emb = sphilberta.encode([latin], convert_to_tensor=True)
    ohg_inputs = mbert_tokenizer([ohg], return_tensors="pt",
                                  truncation=True, max_length=128)
    with torch.no_grad():
        ohg_emb = mbert_model(**ohg_inputs).last_hidden_state[:, 0, :]
        proj_ohg = projection(ohg_emb)
        sim = cosine_similarity(proj_ohg.cpu().numpy(),
                               latin_emb.cpu().numpy())[0][0]

    print(f"\nLatin: {latin[:50]}...")
    print(f"OHG:   {ohg[:50]}...")
    print(f"SPhilBERTa similarity: {sim:.4f}")


Latin: quoniam exaudita est deprecatio tua...
OHG:   wanta ist gibét thinaz fon drúhtine gihórtaz...
SPhilBERTa similarity: 0.8244

Latin: Et dixit Zacharias ad angelum...
OHG:   Thó sprah ther bíscof, harto fóraht er mo thoh...
SPhilBERTa similarity: 0.7399
